<!-- WBPROJ_PATCHED -->
# India — *Made in Heaven* S2 (YouTube) pipeline

End-to-end pipeline used to produce the canonical India datasets:

1. **Filter** raw YouTube comments → `data/interim/india/MIH_S2_cleaned.xlsx`
2. **Embedding-based rerank** for gender relevance → interim files in `archive/india/test_*`
3. **LLM coding** (themes, sentiment, emotions, sexism) → `data/processed/india/MIH_S2_final_dataset.xlsx`
4. **Virality scoring** of YT clips → `data/reach/india/MIH_S2_youtube_virality.csv`
5. **Human-coded ground truth** lives at `data/human_coded/india/MIH_S2_human_coding_extended.xlsx`

Path constants in legacy cells have been rewritten to point at the canonical
locations in this repo. Cells assume the notebook is run from `notebooks/`.
Install dependencies once: `pip install -r requirements.txt` (from repo root).


In [ ]:
import pandas as pd
import re

Dependencies are pinned in `requirements.txt`. Install once from the repo root:

```bash
pip install -r requirements.txt
```


In [ ]:
import pandas as pd
import re

# Path to your dataset
file_path = "../data/raw/india/MIH_S2_full_data.xlsx"

# Load Excel file
df = pd.read_excel(file_path)

# Step 1: Filter ONLY for Made in Heaven Season 2 (different variations)
pattern = r"(made\s*in\s*heaven\s*season\s*2|mih\s*season\s*2|mih\s*s2|made\s*in\s*heaven\s*s2|made\s*in\s*heaven\s*season\s*two)"

mih_df = df[df['title'].str.contains(pattern, case=False, na=False)].copy()

# Step 2: Clean comments
def clean_comment(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()  # normalize casing
    # remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    # remove excessive whitespace
    text = re.sub(r"\s+", " ", text).strip()
    # preserve Hinglish/Hindi tokens + emojis
    return text

mih_df['clean_comment'] = mih_df['comment'].apply(clean_comment)

# Display stats + sample
print("Original dataset size:", len(df))
print("Filtered MIH Season 2 dataset size:", len(mih_df))

mih_df[['title', 'comment', 'clean_comment']].head(10)

In [ ]:
output_path = "../data/interim/india/MIH_S2_cleaned.xlsx"
mih_df.to_excel(output_path, index=False)

print(f"Filtered dataset saved to: {output_path}")

Dependencies are pinned in `requirements.txt`. Install once from the repo root:

```bash
pip install -r requirements.txt
```


Dependencies are pinned in `requirements.txt`. Install once from the repo root:

```bash
pip install -r requirements.txt
```


In [ ]:
import openai
print(openai.__version__)

In [ ]:
import openai
openai.api_key = os.getenv("OPENAI_API_KEY")

In [ ]:
from openai import OpenAI

client = OpenAI(api_key="REDACTED_KEY")

response = client.chat.completions.create(
    model="gpt-5.1",
    messages=[{"role": "user", "content": "Say: My API key is working!"}]
)

print(response.choices[0].message.content)

In [ ]:
import os
import numpy as np
import pandas as pd
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity
import json
import time

client = OpenAI(api_key="REDACTED_KEY")

INPUT_FILE = "../data/interim/india/MIH_S2_cleaned.xlsx"
FILTERED_FILE = "../archive/india/test_MIH_S2_gender_filtered_rerank.xlsx"
FINAL_FILE = "../archive/india/test_MIH_S2_gender_classified.xlsx"

df = pd.read_excel(INPUT_FILE)
df = df.dropna(subset=['comment']).reset_index(drop=True)
print(f"Loaded {len(df)} comments.")

gender_keywords = [

    "caste","Dalit","Brahmin","norms","social norms","cultural norms","stigma","taboo",
    "ritual","ceremony","wedding","marriage","bride","groom","dowry","bride price",
    "dowry harassment","dowry death","arranged marriage","love marriage","child marriage",
    "inter-caste marriage","social boycott","kinship","family pressure","marriage pressure",
    "community elders","traditional values","custom","sin","purity","honor","family honor",
    "reputation","shame","respectability","public image","hypocrisy","societal judgment",
    "samaj","izzat","badnaam","rishta","shaadi","vivaah","ghar ki izzat","samajik daag",
    "samajik dabav","khandaan","parampara","reeti-riwaaz","biradari","jati","jaat",
    "ladki dekhna","rishta todna","log kya kahenge","izzat ka sawal","beizzati",
    "sharm aani chahiye","mooh kala",
    

    "gender equality","gender inequality","gender roles","patriarchy","male dominance",
    "male entitlement","wife duties","obedience","submission","purity","virginity",
    "chastity","victim-blaming","feminism","feminist","women","girls","men","boys",
    "husband","wife","single mother","breadwinner","man of the house","suffocating family",
    "no choice forced","control","supposed to","expected to","as a woman","as a man",
    "women should","women shouldn’t","men should","men shouldn’t",
    "ladki ghar sambhale","pati-parmeshwar","ghar ki laxmi","gharwali","pativrata",
    "sati-savitri","izzat bachaana","ladkiyon ko itna padhana kya zaroori hai",
    "ladkiyon ki maryada","mard ki baat","ladka ladki barabar","ghar ka mard",
    "bahu ki zimmedari","ladki ko chup rehna chahiye","ladki ki izzat ka sawal",


    "abuse","gaslighting","emotional manipulation","verbal abuse","physical violence",
    "coercion","marital rape","silence","trauma","red flags","victimhood",
    "financial control","emotional blackmail","abusive marriage","shattered dreams",
    "loveless marriage","no empathy","suffocation","disappointment","broken trust",
    "normalized abuse","endurance","wilted smile","fear of divorce","social stigma divorce",
    "maar-peet","atyachar","sharirik hinsa","pati ka zulm","ghar todna","chhupna",
    "samaj kya sochega","talaaq","pati ki marzi","roona-dhona","bardasht karna",
    "apmaan","izzat ka sawaal","haat uthana",


    "women’s voice","speaking up","breaking silence","liberation","independence",
    "resilience","strength","agency","choice","self-worth","career","entrepreneur",
    "professional growth","leadership","education","scholarship","economic empowerment",
    "financial independence","upward mobility","breaking stereotypes","fight together",
    "resistance","dignity","pride","girl boss",
    "apni awaaz uthana","khud ki pehchaan","swatantrata","ladkiyan kuch bhi kar sakti hain",
    "apne paon pe khadi","apni zindagi apne faisle","khud kaam karna",
    "ladkiya bojh nahi hain","nari shakti","apni izzat","apni marzi",

    "sexual health","reproductive rights","contraception","family planning",
    "safe sex","unplanned pregnancy","abortion rights","access to healthcare",
    "maternal health","menstrual health","menstrual hygiene","period poverty",
    "reproductive justice","SRHR","mahavari","periods","swachhta","garbh nirodh",
    "garbhpat","garbh niyantran","garbh dharan","maa ban’na","bachcha paida karna",
    "janam dena","swasthya seva",

    "Dalit woman","caste discrimination","caste oppression","caste privilege",
    "Brahmin supremacy","untouchability","caste-based violence","caste honor",
    "stigma caste","family honor linked to caste","dowry as caste marker",
    "jaat-bhed","chhoti jaat","badi jaat","jaat tod vivah","samaajik bahishkar",
    "jaati maryada","jaat ki izzat","ghar ki izzat jaat se judi","jaati-vaati",
    "quota waale","reservations waale",

    "fair skin","dark skin","body shaming","figure","cosmetic pressure",
    "fairness cream","beauty queen","glowing bride","appearance vs reality",
    "cosmetic surgery","societal expectations of beauty","market value",
    "objectification","trophy wife","gora rang","kaala rang","gori dulhan",
    "kaali ladki","sundarta ka pressure","khoobsurat dulhan","sundar bahu",
    "asli khoobsurti","rang pehle dekha jata hai","chehre ki chamak","Fair & Lovely",

    "fear","anger","helplessness","denial","sadness","courage","hope","empowerment",
    "relief","low self-esteem","heartbreak","rage","exhaustion","unshed tears",
    "glowing happiness","beaming pride","resilience","shame","guilt","anxiety",
    "longing","ambition","dignity","defiance","darr","gussa","majboori","nirasha",
    "dukh","himmat","umeed","apmaan","garv","shakti","pyaar","nafrat","aansu",
    "rona","dhokha","dard","sharm"
]

In [ ]:
print("Running embedding-based rerank...")

comments = df["comment"].tolist()

BATCH = 100
comment_embeddings = []
for i in range(0, len(comments), BATCH):
    batch = comments[i:i+BATCH]
    resp = client.embeddings.create(model="text-embedding-3-large", input=batch)
    batch_embs = [r.embedding for r in resp.data]
    comment_embeddings.extend(batch_embs)
    print(f"Processed {i+len(batch)} / {len(comments)} comments")

comment_embeddings = np.array(comment_embeddings, dtype="float32")

query_text = " ".join(gender_keywords)
query_embedding = client.embeddings.create(
    model="text-embedding-3-large",
    input=[query_text]
).data[0].embedding

print("Computing similarities...")
sims = cosine_similarity(comment_embeddings, [query_embedding]).flatten()

top_n = 1000
top_idx = np.argsort(sims)[-top_n:][::-1]
filtered_df = df.iloc[top_idx].reset_index(drop=True)

print(f"Original: {len(df)} | Retained top {top_n}: {len(filtered_df)}")
filtered_df.to_excel(FILTERED_FILE, index=False)

themes = [
    "India Local Culture","Gender Norms & Dynamics","Gender-Based Violence",
    "Economic Empowerment","Sexual & Reproductive Health",
    "Caste / Intersectional","Body & Beauty Standards","Emotional States"
]

def classify_comment(comment):
    prompt = f"""
    Classify this comment into one or more of these themes: {themes}.
    Also assign sentiment: Positive, Negative, or Neutral.
    Return JSON only.
    Comment: "{comment}"
    """
    try:
        resp = client.chat.completions.create(
            model="gpt-5.1",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        print("Classification error:", e)
        return json.dumps({"themes":[],"sentiment":"Error"})

BATCH_SIZE = 500
classified_rows = []

for start in range(0, len(filtered_df), BATCH_SIZE):
    batch = filtered_df.iloc[start:start+BATCH_SIZE].copy()
    print(f"Processing batch {start//BATCH_SIZE+1}...")
    batch["classification"] = batch["comment"].apply(classify_comment)
    classified_rows.append(batch)
    # Save progress incrementally
    pd.concat(classified_rows).to_excel(FINAL_FILE, index=False)
    print(f"Saved progress up to row {start+len(batch)}")

# Final merge
final_df = pd.concat(classified_rows).reset_index(drop=True)

def parse_classification(row):
    try:
        data = json.loads(row)
        return pd.Series([",".join(data.get("themes", [])), data.get("sentiment", "")])
    except:
        return pd.Series(["", ""])

final_df[["themes","sentiment"]] = final_df["classification"].apply(parse_classification)
final_df.to_excel(FINAL_FILE, index=False)

print(f"✅ Done! Final classified dataset saved at: {FINAL_FILE}")

In [ ]:
import pandas as pd
import json

# Load your classified dataset
df = pd.read_excel("../archive/india/test_MIH_S2_gender_classified.xlsx")

# Parse classification JSON
def parse_classification(row):
    try:
        clean = row.strip("`")  # remove markdown formatting
        data = json.loads(clean.replace("json", "").strip())
        return pd.Series([",".join(data.get("themes", [])), data.get("sentiment", "")])
    except Exception as e:
        return pd.Series(["", ""])

df[["themes","sentiment"]] = df["classification"].apply(parse_classification)

# Save back
df.to_excel("../archive/india/test_MIH_S2_gender_classified_fixed.xlsx", index=False)
print("✅ Fixed file saved with themes & sentiment columns filled.")

Dependencies are pinned in `requirements.txt`. Install once from the repo root:

```bash
pip install -r requirements.txt
```


In [ ]:
import pandas as pd

# Paths
classified_file = "../archive/india/test_MIH_S2_gender_classified.xlsx"
codebook_file   = "../archive/misc/wb_social_media_coding_45rows.xlsx"
output_file     = "../data/interim/india/MIH_S2_codebook_ready.xlsx"

# Load datasets
df_classified = pd.read_excel(classified_file)
df_codebook   = pd.read_excel(codebook_file)

# Keep only needed columns from classified dataset
df_classified = df_classified[["comment", "themes", "sentiment"]].copy()
df_classified = df_classified.reset_index(drop=True)

# Prepare merged structure with codebook columns
merged = pd.DataFrame({
    "Comment ID": df_classified.index + 1,
    "Comment Text": df_classified["comment"],
    "Mentions Media?": "",
    "What Content?": "",
    "Attitude": "",
    "Mentions Gender?": "",
    "Themes": df_classified["themes"],
    "Sentiment": df_classified["sentiment"],
    "Emotions": "",
    "Other Notes": ""
})

# Save to Excel
merged.to_excel(output_file, index=False)

print(f"✅ Codebook-ready dataset saved at: {output_file}")

In [ ]:
FILE = "../data/human_coded/india/MIH_S2_human_coding_extended.xlsx"
OUTPUT = "../data/interim/india/MIH_S2_yt_human_coding_with_emotions.xlsx"

# --- Load dataset ---
df = pd.read_excel(FILE)

# Ensure target column exists
if "Here are the emotions detected by LLM: X." not in df.columns:
    df["Here are the emotions detected by LLM: X."] = ""

# --- Function to classify emotions ---
def detect_emotions(comment):
    prompt = f"""
    Identify the emotions expressed in this comment. 
    Choose only from: Contempt, Anger, Disgust, Sadness, Shame, Embarrassment, Guilt, Compassion, Pride, No emotion.
    Return as a JSON list, e.g.: {{"emotions": ["Anger","Sadness"]}}
    
    Comment: "{comment}"
    """
    try:
        resp = client.chat.completions.create(
            model="gpt-5.1",
            messages=[{"role":"user","content": prompt}],
            temperature=0
        )
        content = resp.choices[0].message.content.strip()
        data = json.loads(content) if content.startswith("{") else {"emotions":[content]}
        return ", ".join(data.get("emotions", []))
    except Exception as e:
        return "Error"

# --- Apply detection ---
for i, row in df.iterrows():
    if pd.isna(row["Here are the emotions detected by LLM: X."]) or row["Here are the emotions detected by LLM: X."] == "":
        df.at[i, "Here are the emotions detected by LLM: X."] = detect_emotions(str(row["Comment Text"]))
        print(f"Processed {i+1}/{len(df)}")

# --- Save result ---
df.to_excel(OUTPUT, index=False)
print(f"✅ Done! File saved at: {OUTPUT}")

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

# --- Load dataset ---
file_path = "../data/interim/india/MIH_S2_cleaned.xlsx"
df = pd.read_excel(file_path)

# --- Convert date and remove timezone info ---
df['date'] = pd.to_datetime(df['date'], errors='coerce').dt.tz_localize(None)

# --- Compute days since upload ---
today = datetime.now()
df['days_since_upload'] = (today - df['date']).dt.days
df['days_since_upload'] = df['days_since_upload'].replace(0, 1)  # avoid divide-by-zero

# --- Define Virality Metrics ---
# 1. Reach Factor (scale but dampened)
df['ReachFactor'] = np.log1p(df['viewCount'])

# 2. Engagement Efficiency (likes per view)
df['EngagementEfficiency'] = df['likes'] / df['viewCount'].replace(0, 1)

# 3. Velocity Factor (likes per day)
df['VelocityFactor'] = df['likes'] / df['days_since_upload']

# 4. Virality Score (multiplicative model)
df['ViralityScore'] = df['ReachFactor'] * df['EngagementEfficiency'] * df['VelocityFactor']

# --- Sort videos by Virality Score ---
df_sorted = df.sort_values('ViralityScore', ascending=False)

# --- Save report ---
output_path = "../data/reach/india/MIH_S2_youtube_virality.csv"
df_sorted.to_csv(output_path, index=False)

# --- Preview top 10 ---
print(f"Improved virality report saved to: {output_path}")
print(df_sorted[['title','url','viewCount','likes',
                 'ReachFactor','EngagementEfficiency','VelocityFactor','ViralityScore']].head(10))

In [ ]:
# --- Imports ---
import pandas as pd, time, json
from tqdm import tqdm
from openai import OpenAI

# --- Initialize OpenAI client ---
# ⚠️ Replace with your actual OpenAI API key
client = OpenAI(api_key="REDACTED_KEY")

# --- Load dataset ---
file_path = "../archive/india/MIH_S2_final_dataset__no_dup.xlsx"
df = pd.read_excel(file_path)
df = df.dropna(subset=["comment"]).reset_index(drop=True)
print(f"Loaded {len(df)} comments.")

# --- Function to detect sexist / derogatory content ---
def detect_sexism(comment, retries=2):
    """
    Detect whether a comment contains sexist or derogatory language.
    Returns a tuple (flag, quote).
    """
    prompt = f"""
    You are analyzing online comments for harmful or biased language.

    Question 1: Does the comment include any sexist, misogynistic, or derogatory language or tone?
    Question 2: If YES, provide a short direct quote (a few words) from the comment that shows it.

    Respond ONLY with valid JSON, e.g.:
    {{"sexist_flag": "Yes", "sexist_quote": "women shouldn't drive"}}
    or
    {{"sexist_flag": "No", "sexist_quote": ""}}

    Comment: "{comment}"
    """

    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model="gpt-5.1",
                messages=[{"role": "user", "content": prompt}],
                temperature=0
            )
            text = resp.choices[0].message.content.strip()
            text = text.replace("```json", "").replace("```", "").strip()
            data = json.loads(text)
            flag = data.get("sexist_flag", "").capitalize()
            quote = data.get("sexist_quote", "").strip()
            if flag not in ["Yes", "No"]:
                raise ValueError("Invalid flag value")
            return flag, quote
        except Exception as e:
            print(f"⚠️ Error attempt {attempt+1}: {e}")
            time.sleep(0.5)
    return "Error", ""

# --- Apply to each comment ---
sexist_flags, sexist_quotes = [], []

for comment in tqdm(df["comment"], desc="Analyzing comments for sexism"):
    flag, quote = detect_sexism(comment)
    sexist_flags.append(flag)
    sexist_quotes.append(quote)
    time.sleep(0.3)  # small delay to avoid rate limits

# --- Add new columns ---
df["sexist_flag"] = sexist_flags
df["sexist_quote"] = sexist_quotes

# --- Save results ---
output_path = "../archive/india/MIH_S2_final_dataset__with_sexism_pre_merge.xlsx"
df.to_excel(output_path, index=False)

print(f"\n✅ Done! File saved at:\n{output_path}")

In [ ]:
import pandas as pd

# Path to your file
file_path = "../archive/india/MIH_S2_final_dataset__with_sexism_pre_merge.xlsx"

# Load dataset
df = pd.read_excel(file_path)

# Combine 'sexist_flag' and 'sexist_quote' with a comma (handle missing values gracefully)
df["sexism_summary"] = df.apply(
    lambda row: f"{row['sexist_flag']}, {row['sexist_quote']}"
    if pd.notnull(row['sexist_quote']) and str(row['sexist_quote']).strip() != ""
    else str(row['sexist_flag']),
    axis=1
)

# Save merged version
output_path = "../data/processed/india/MIH_S2_final_dataset.xlsx"
df.to_excel(output_path, index=False)

print(f"✅ Done! Merged file saved at:\n{output_path}")

Dependencies are pinned in `requirements.txt`. Install once from the repo root:

```bash
pip install -r requirements.txt
```


In [ ]:
import pandas as pd
from langdetect import detect, DetectorFactory
from tqdm import tqdm
import re

# === CONFIG ===
input_path = "../data/human_coded/india/MIH_S2_human_coding_extended.xlsx"
output_path = "../archive/india/MIH_S2_final_200_mix.xlsx"
comment_col = "Comment Text"

# === SETUP ===
DetectorFactory.seed = 0
tqdm.pandas()

# === LOAD DATA ===
df = pd.read_excel(input_path)
df = df[df[comment_col].notna()].copy()
df[comment_col] = df[comment_col].astype(str)

# === SAFE LANGUAGE DETECTION ===
def detect_lang_safe(text):
    try:
        return detect(text)
    except:
        return "unknown"

print("Detecting language... please wait.")
df["lang"] = df[comment_col].progress_apply(detect_lang_safe)

# === IDENTIFY HINGLISH HEURISTICALLY ===
# Hinglish = Latin script but contains Hindi words like 'nahi', 'acha', 'ladki', etc.
hinglish_keywords = ["nahi", "acha", "samaj", "ladki", "ladka", "pata", "bol", "bata", "kya", "kaise", "log", "bahut", "kar", "hai", "tha", "thi"]
def is_hinglish(text):
    text_low = text.lower()
    if re.search(r"[ऀ-ॿ]", text_low):  # skip if it's actually Devanagari
        return False
    return any(word in text_low for word in hinglish_keywords)

df["is_hinglish"] = df[comment_col].apply(is_hinglish)

# === FILTER BY LANGUAGE ===
df_en = df[df["lang"] == "en"]
df_hi = df[df["lang"] == "hi"]
df_hinglish = df[df["is_hinglish"] & (df["lang"] != "hi") & (df["lang"] != "unknown")]

# === SAMPLE TARGET COUNTS ===
df_en_sample = df_en.sample(min(180, len(df_en)), random_state=42)
df_hi_sample = df_hi.sample(min(10, len(df_hi)), random_state=42)
df_hinglish_sample = df_hinglish.sample(min(10, len(df_hinglish)), random_state=42)

# === COMBINE ALL ===
final_df = pd.concat([df_en_sample, df_hi_sample, df_hinglish_sample]).sample(frac=1, random_state=42).reset_index(drop=True)

# === SAVE TO EXCEL ===
final_df.to_excel(output_path, index=False)

print(f"✅ Saved {len(final_df)} comments (180 English, 10 Hindi, 10 Hinglish) to:\n{output_path}")

Dependencies are pinned in `requirements.txt`. Install once from the repo root:

```bash
pip install -r requirements.txt
```


In [ ]:
import re
from collections import Counter, defaultdict
import numpy as pd
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from docx import Document
from docx.shared import Inches

# ------------- CONFIG -------------
FILE_PATH = r"../data/human_coded/india/MIH_S2_human_coding_extended.xlsx"
OUT_DIR = Path(FILE_PATH).parent / "mih_figures_pro"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CREATE_DOCX = True
DOCX_PATH = OUT_DIR / "MIH_S2_Visuals.docx"

# ------------- LOAD -------------
df = pd.read_excel(FILE_PATH)
colmap = {c.lower().strip(): c for c in df.columns}

def try_get(*cands):
    for cand in cands:
        k = cand.lower().strip()
        if k in colmap:
            return colmap[k]
    return None

COMMENT_COL = try_get("Comment Text", "comment", "text", "comments")
THEMES_COL = try_get("Themes", "Theme", "topics")
SENTIMENT_TEXT_COL = try_get("Sentiment")
SENTIMENT_CODED_COL = try_get(
    " If Y: what is the general attitude toward this media content? 1=Positive/2=Neutral/3=Negative/4=Mixed/5=Unclear",
    "If Y: what is the general attitude toward this media content? 1=Positive/2=Neutral/3=Negative/4=Mixed/5=Unclear",
)
NORMS_ATT_COL = try_get(
    "If Y: What is the general attitude expressed toward discussed ...nder norms? 1=Positive/2=Neutral/3=Negative/4=Mixed/5=Unclear",
    "If Y: What is the general attitude expressed toward discussed gender norms? 1=Positive/2=Neutral/3=Negative/4=Mixed/5=Unclear",
)

if not COMMENT_COL:
    raise ValueError("Comment column not found. Update COMMENT_COL mapping.")

comments = df[COMMENT_COL].dropna().astype(str)

# ------------- HELPERS -------------
plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 200,
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

def number_to_sentiment(n):
    try:
        n = int(n)
    except Exception:
        return None
    return {1:"Positive", 2:"Neutral", 3:"Negative", 4:"Mixed", 5:"Unclear"}.get(n)

def number_to_norm_stance(n):
    try:
        n = int(n)
    except Exception:
        return None
    # Positive toward "gender norms" => Supporting tradition; Negative => Challenging; others => Discussing
    return {1:"Supporting", 2:"Discussing", 3:"Challenging", 4:"Discussing", 5:"Discussing"}.get(n)

def split_themes(val):
    if pd.isna(val): return []
    parts = re.split(r"[;,/|]+", str(val))
    return [p.strip() for p in parts if p.strip()]

# Sentiment labels
if SENTIMENT_TEXT_COL:
    sentiment = (
        df[SENTIMENT_TEXT_COL].astype(str).str.strip().str.capitalize().replace("", pd.NA).dropna()
    )
elif SENTIMENT_CODED_COL:
    sentiment = df[SENTIMENT_CODED_COL].map(number_to_sentiment).dropna()
else:
    # lightweight heuristic fallback
    pos = re.compile(r"\b(great|amazing|powerful|love|brilliant|excellent|respect)\b", re.I)
    neg = re.compile(r"\b(terrible|awful|hate|nonsense|trash|worst|cringe)\b", re.I)
    tmp = []
    for t in comments:
        if pos.search(t): tmp.append("Positive")
        elif neg.search(t): tmp.append("Negative")
        else: tmp.append("Neutral")
    sentiment = pd.Series(tmp)

sent_counts = sentiment.value_counts().to_dict()

# Norm stance labels
if NORMS_ATT_COL:
    norm_stance = df[NORMS_ATT_COL].map(number_to_norm_stance).dropna()
else:
    # fallback keywords
    support_kw = re.compile(r"\b(traditional values|sanskari|duty|obey|family honor|should\s+(?=.*wife|woman))\b", re.I)
    chall_kw = re.compile(r"\b(patriarchy|double standards|misogyny|toxic|dowry|freedom|autonomy|equality)\b", re.I)
    tmp = []
    for t in comments:
        if chall_kw.search(t): tmp.append("Challenging")
        elif support_kw.search(t): tmp.append("Supporting")
        else: tmp.append("Discussing")
    norm_stance = pd.Series(tmp)

stance_counts = norm_stance.value_counts().to_dict()

# Themes
if THEMES_COL:
    theme_series = df[THEMES_COL].apply(split_themes)
    all_themes = [th for lst in theme_series for th in lst]
    theme_counts = Counter(all_themes)
else:
    # keyword fallback
    theme_defs = {
        "Marriage & Gender Roles": r"\b(marriage|wedding|wife|husband|bride|groom|compromise|housewife|breadwinner)\b",
        "Dowry & Finance": r"\b(dowry|ghar\s*jamai|dahej|money|wealth|status)\b",
        "Feminism & Empowerment": r"\b(feminist|feminism|empowerment|agency|independence|liberation)\b",
        "LGBTQ+ / Sexuality": r"\b(gay|lgbt|queer|same[- ]sex|homophobia|closet|pride)\b",
        "Class & Privilege": r"\b(rich|privilege|class|elite|luxury|status)\b",
        "Hypocrisy / Double Standards": r"\b(hypocrisy|double standards|two[- ]faced)\b",
    }
    theme_counts = Counter()
    for t in comments:
        for theme, pat in theme_defs.items():
            if re.search(pat, t, re.I):
                theme_counts[theme] += 1

# Top N themes for visuals
TOP_N = 10
top_themes = dict(theme_counts.most_common(TOP_N))

# ---------- 1) SENTIMENT DONUT ----------
fig = plt.figure(figsize=(6.2, 6.2))
labels = list(sent_counts.keys())
values = list(sent_counts.values())
plt.pie(values, labels=labels, autopct="%1.0f%%", startangle=90)
# donut hole
centre_circle = plt.Circle((0,0), 0.60, fc="white")
fig.gca().add_artist(centre_circle)
plt.title("Sentiment Distribution — MIH S2")
sent_png = OUT_DIR / "01_sentiment_donut.png"
plt.tight_layout()
plt.savefig(sent_png, bbox_inches="tight")
plt.close()

# ---------- 2) GENDER NORM STANCE (HORIZONTAL BAR + LABELS) ----------
fig = plt.figure(figsize=(7.2, 4.5))
items = sorted(stance_counts.items(), key=lambda x: x[1])  # ascending for clean bars
labels = [k for k,_ in items]
vals = [v for _,v in items]
ypos = range(len(labels))
plt.barh(list(ypos), vals)
plt.yticks(list(ypos), labels)
for y, v in enumerate(vals):
    plt.text(v + max(vals)*0.01, y, str(v), va="center")
plt.xlabel("Count")
plt.title("Attitudes Toward Traditional Gender Norms — MIH S2")
stance_png = OUT_DIR / "02_gender_norm_stance_barh.png"
plt.tight_layout()
plt.savefig(stance_png, bbox_inches="tight")
plt.close()

# ---------- 3) TOP THEMES LOLLIPOP (READABLE, MINIMAL) ----------
fig = plt.figure(figsize=(8.8, 5.2))
items = sorted(top_themes.items(), key=lambda x: x[1])  # ascending
labels = [k for k,_ in items]
vals = [v for _,v in items]
y = range(len(labels))
# stems
for yi, v in zip(y, vals):
    plt.plot([0, v], [yi, yi], linewidth=2)
# markers
plt.scatter(vals, list(y), s=70)
plt.yticks(list(y), labels)
plt.xlabel("Mentions (count)")
plt.title("Top Themes Frequency — MIH S2")
themes_png = OUT_DIR / "03_themes_lollipop.png"
plt.tight_layout()
plt.savefig(themes_png, bbox_inches="tight")
plt.close()

# ---------- 4) THEME × SENTIMENT HEATMAP ----------
# Build cross-tab
def theme_presence(series, theme_list):
    rows = []
    for ths in series:
        if pd.isna(ths):
            rows.append(set())
        else:
            rows.append(set(split_themes(ths)))
    return rows

if THEMES_COL:
    theme_rows = theme_presence(df[THEMES_COL], list(top_themes.keys()))
    theme_names = list(top_themes.keys())
else:
    # rebuild a boolean matrix from keyword fallback
    theme_names = list(top_themes.keys())
    pats = {t: re.compile(r".*", re.I) for t in []}  # not used
    theme_rows = []
    for txt in comments:
        present = set()
        for th in theme_names:
            # crude membership from earlier counts: we re-check with substring of theme label words
            key = th.split("&")[0].split("/")[0].strip().split()[0]
            if re.search(re.escape(key), txt, re.I):
                present.add(th)
        theme_rows.append(present)

sent_norm = sentiment.fillna("Unclear").astype(str)
heat = pd.DataFrame(0, index=theme_names, columns=sorted(sent_norm.unique()))
for i, thset in enumerate(theme_rows):
    s = sent_norm.iloc[i] if i < len(sent_norm) else "Unclear"
    for th in thset:
        if th in heat.index and s in heat.columns:
            heat.loc[th, s] += 1

fig = plt.figure(figsize=(9, 6))
ax = plt.gca()
im = ax.imshow(heat.values, aspect="auto")
ax.set_xticks(range(len(heat.columns)))
ax.set_xticklabels(heat.columns, rotation=45, ha="right")
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels(heat.index)
plt.title("Themes × Sentiment — MIH S2")
# annotate cells
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        val = heat.values[i, j]
        if val:
            ax.text(j, i, str(val), ha="center", va="center")
heat_png = OUT_DIR / "04_theme_by_sentiment_heatmap.png"
plt.tight_layout()
plt.savefig(heat_png, bbox_inches="tight")
plt.close()

# ---------- 5) (OPTIONAL) STACKED BAR: NORM STANCE × SENTIMENT ----------
stack_png = None
if NORMS_ATT_COL and (SENTIMENT_TEXT_COL or SENTIMENT_CODED_COL):
    # align lengths
    ndf = pd.DataFrame({"stance": norm_stance, "sent": sentiment}).dropna()
    pivot = pd.crosstab(ndf["stance"], ndf["sent"]).reindex(
        ["Supporting", "Discussing", "Challenging"]
    ).fillna(0).astype(int)

    fig = plt.figure(figsize=(7.6, 5.0))
    bottom = None
    for col in pivot.columns:
        plt.bar(pivot.index, pivot[col].values, bottom=bottom, label=col)
        bottom = (pivot[col].values if bottom is None else bottom + pivot[col].values)
    plt.ylabel("Count")
    plt.title("Norm Stance × Sentiment — MIH S2")
    plt.legend(title="Sentiment", ncol=2)
    stack_png = OUT_DIR / "05_normstance_by_sentiment_stacked.png"
    plt.tight_layout()
    plt.savefig(stack_png, bbox_inches="tight")
    plt.close()

print("Saved:")
print("-", sent_png)
print("-", stance_png)
print("-", themes_png)
print("-", heat_png)
if stack_png:
    print("-", stack_png)

# --------- OPTIONAL: assemble visuals into a simple Word doc ----------
if CREATE_DOCX:
    doc = Document()
    doc.add_heading("Made in Heaven S2 — Visual Insights", level=1)

    doc.add_heading("Sentiment Distribution (Donut)", level=2)
    doc.add_picture(str(sent_png), width=Inches(6.2))

    doc.add_heading("Attitudes Toward Traditional Gender Norms", level=2)
    doc.add_picture(str(stance_png), width=Inches(6.2))

    doc.add_heading("Top Themes Frequency (Lollipop Chart)", level=2)
    doc.add_picture(str(themes_png), width=Inches(6.2))

    doc.add_heading("Themes × Sentiment (Heatmap)", level=2)
    doc.add_picture(str(heat_png), width=Inches(6.2))

    if stack_png:
        doc.add_heading("Norm Stance × Sentiment (Stacked Bar)", level=2)
        doc.add_picture(str(stack_png), width=Inches(6.2))

    doc.save(DOCX_PATH)
    print(f"Word doc saved to: {DOCX_PATH}")

In [ ]:
import pandas as pd
from openai import OpenAI
from pathlib import Path
import time

# ---- CONFIG ----
FILES = [
    r"../data/human_coded/india/MIH_S2_human_coding_extended.xlsx",
    r"../archive/india/MIH_S2_yt_for_llm_coding__193rows.xlsx"
]
MODEL = "gpt-5.1"
COMMENT_COL_CANDIDATES = ["Comment Text", "comment", "text", "comments"]
SLEEP_BETWEEN_CALLS = 0.4  # seconds between API calls

# ---- SETUP ----
client = OpenAI()

def detect_language(text):
    """Detect if text is English, Hindi, or Hinglish using GPT-5.1-mini."""
    prompt = f"""
Identify whether the following text is written primarily in **English**, **Hindi**, or **Hinglish** 
(mixed English + Hindi written in Latin script). Respond with exactly one word: English, Hindi, or Hinglish.

Text:
{text[:500]}
"""
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "You are a precise language classifier for short social media comments."},
                {"role": "user", "content": prompt}
            ],
            temperature=0
        )
        lang = response.choices[0].message.content.strip()
        return lang if lang in ["English", "Hindi", "Hinglish"] else "Unknown"
    except Exception as e:
        print("Error:", e)
        return "Error"

# ---- MAIN LOOP ----
for file_path in FILES:
    print(f"\n🔍 Processing file: {file_path}")
    df = pd.read_excel(file_path)
    
    # Identify comment column
    colmap = {c.lower().strip(): c for c in df.columns}
    comment_col = None
    for cand in COMMENT_COL_CANDIDATES:
        if cand.lower().strip() in colmap:
            comment_col = colmap[cand.lower().strip()]
            break
    if not comment_col:
        raise ValueError(f"No recognizable comment column found in {file_path}. Please update COMMENT_COL_CANDIDATES.")
    
    # Prepare comments
    df[comment_col] = df[comment_col].astype(str).fillna("")
    comments = df[comment_col].tolist()

    # Detect language per comment
    languages = []
    for i, text in enumerate(comments):
        lang = detect_language(text)
        languages.append(lang)
        if (i + 1) % 10 == 0:
            print(f"Processed {i + 1}/{len(comments)} → {lang}")
        time.sleep(SLEEP_BETWEEN_CALLS)

    df["language"] = languages

    # Build new filename
    path_obj = Path(file_path)
    new_name = path_obj.stem + "____lang.xlsx"
    out_path = path_obj.with_name(new_name)

    # Save updated file
    df.to_excel(out_path, index=False)
    
    # Print summary
    counts = df["language"].value_counts()
    print(f"\n📊 Language counts for {path_obj.name}:")
    print(counts.to_string())
    print(f"✅ Saved file with language column → {out_path}")

In [ ]:
import pandas as pd

# Path to the processed file (already has 'language' column)
file_path = r"../archive/india/MIH_S2_yt_human_coding_results__lang.xlsx"

# Load and display counts
df = pd.read_excel(file_path)
if "language" not in df.columns:
    raise ValueError("No 'language' column found. Make sure the ____lang version of the file is loaded.")

print(f"📊 Language distribution for {file_path}:")
print(df["language"].value_counts().to_string())